# Критерии оценки ДЗ

> **Версия LanceDB: `lancedb<0.20` !!!**

Делаете прогоны на test_dataset, выбиваете максимум по метрикам. Я буду проверять на eval_dataset

---

## Базовые требования (4 балла)

| # | Критерий | Баллы |
|---|----------|-------|
| 1 | Описан процесс подготовки данных (`data_preporation.py`, `embedder.py`) | 1 |
| 2 | Описаны методы/функции для работы с векторной БД в `lance_db.py` (обязательно `read_all`) | 1 |
| 3 | Данные записаны в БД и можно выполнить `read_all` | 1 |
| 4 | Описан флоу retrieve и agent (`agent.py`, `retriever.py`) | 1 |

---

## Метрики качества (до 9 баллов)

### F1@K

| Порог    | Баллы |
|----------|-------|
| > 0.50   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### NDCG@K

| Порог    | Баллы |
|----------|-------|
| > 0.65   | 1     |
| > 0.75   | 2     |
| > 0.90   | 3     |

### Faithfulness

| Порог    | Баллы |
|----------|-------|
| > 0.75   | 1     |
| > 0.85   | 2     |
| > 0.95   | 3     |

---

## Лог эволюции (до 2 баллов)

Оценивается индивидуально на основании полноты информации в `EVOLUTION.md`.

---

## Штрафы

| Условие | Штраф |
|---------|-------|
| Суммарный объём всех передаваемых для генерации чанков > 6000 символов | −100500 баллов |
| Использовали дополнительные библиотеки и не положили requirements.txt | −100500 баллов |

---

**Максимум: 15 баллов** (4 базовых + 9 метрики + 2 эволюция)

# Валидация RAG-агента через модуль `evaluation.py`

Демонстрация полного пайплайна оценки:
1. Подключаемся к векторному хранилищу и создаём агента
2. Загружаем тестовый датасет
3. Прогоняем агента на всех вопросах (многопоточно)
4. Вызываем `evaluate()` — получаем DataFrame с 7 метриками
5. Анализируем результаты

---
## 0. Подготовка

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.11.6 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [2]:
!git clone https://github.com/GonnaMakeYouCry/HSE-Agent-Systems_2026.git


fatal: destination path 'HSE-Agent-Systems_2026' already exists and is not an empty directory.


In [3]:
%cd HSE-Agent-Systems_2026


/content/HSE-Agent-Systems_2026


In [4]:
!uv sync


Resolved 154 packages in 2ms
Checked 151 packages in 4ms


In [5]:
!cp .env.example .env


In [6]:
import warnings

warnings.filterwarnings("ignore")

import sys
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm


def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists():
            return p
    return start


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import settings
from openai import OpenAI

In [7]:
client = OpenAI(
    base_url="https://api.polza.ai/api/v1",
    api_key=settings.polza_ai_api_key,
)

MODEL = "openai/gpt-4o-mini"

---
## 1. Подключаемся к векторному хранилищу и создаём агента

In [8]:
%cd lectures/lecture_4

/content/HSE-Agent-Systems_2026/lectures/lecture_4


In [9]:
!pip install -q "lancedb<0.20" openai pypdf pymupdf Pillow pandas openpyxl tqdm langchain-text-splitters pydantic pydantic-settings ragas nest-asyncio
!pip install -q "requests==2.32.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [10]:
# !rm -rf lance_db/vectorstore

In [11]:
from pathlib import Path

import lancedb

from lance_db import build_vectorstore

VECTORSTORE_DIR = "lance_db/vectorstore"
TABLE_NAME = "chunks"

db_path = Path(VECTORSTORE_DIR)
db = lancedb.connect(VECTORSTORE_DIR)

table_exists = False
try:
    table_exists = TABLE_NAME in set(db.table_names())
except Exception:
    table_exists = False

if table_exists:
    table = db.open_table(TABLE_NAME)
else:
    table = build_vectorstore(
        client=client,
        dataset_dir="dataset",
        vectorstore_dir=VECTORSTORE_DIR,
        table_name=TABLE_NAME,
        embedding_model="text-embedding-3-small",
        force_rebuild=True,
    )
    db = lancedb.connect(VECTORSTORE_DIR)

table = db.open_table(TABLE_NAME)

print(f"✅ Таблица '{TABLE_NAME}' открыта: {table.count_rows()} строк")

✅ Таблица 'chunks' открыта: 21 строк


In [12]:
import lancedb

db = lancedb.connect("lance_db/vectorstore")
table = db.open_table("chunks")
df = table.head(table.count_rows()).to_pandas()

print("rows:", len(df))
print("\ncolumns:", list(df.columns))

print("\nby year:")
print(df["year"].value_counts(dropna=False).sort_index())

print("\nby source:")
print(df["source"].value_counts(dropna=False))

display(df[["chunk_id", "source", "year", "page", "text"]].head(21))

rows: 21

columns: ['chunk_id', 'source', 'source_file', 'year', 'page', 'text', 'raw_text', 'visual_text', 'has_visual_text', 'search_text', 'text_len', 'vector']

by year:
year
2019     8
2025    13
Name: count, dtype: int64

by source:
source
Consumer_sentiment_2Q2025.pdf           13
Potrebitelskie_ozhidaniya_2_2019.pdf     8
Name: count, dtype: int64


,chunk_id,source,year,page,text
0,0,Consumer_sentiment_2Q2025.pdf,2025,1,ПОТРЕБИТЕЛЬСКИЕ НАСТРОЕНИЯ НАСЕЛЕНИЯВО II КВАР...
1,1,Consumer_sentiment_2Q2025.pdf,2025,2,"2 \n \n \nАвторы: \nОстапкович Г.В., Лола И.С...."
2,2,Consumer_sentiment_2Q2025.pdf,2025,3,3 \n \nЦентр конъюнктурных исследований Инстит...
3,3,Consumer_sentiment_2Q2025.pdf,2025,4,4 \n \nнегативного на позитивный – крайне редк...
4,4,Consumer_sentiment_2Q2025.pdf,2025,5,"5 \n \nСреди пяти компонент, входящих в структ..."
5,5,Consumer_sentiment_2Q2025.pdf,2025,6,6 \n \nтрлн. руб. в виде дохода от собственнос...
6,6,Consumer_sentiment_2Q2025.pdf,2025,7,"7 \n \nроста в указанной группе, его значение ..."
7,7,Consumer_sentiment_2Q2025.pdf,2025,8,"8 \n \nТак, в период «тучных» лет для российск..."
8,8,Consumer_sentiment_2Q2025.pdf,2025,9,9 \n \nÐàñïðåäåëåíèå îòâåòîâ íà îòäåëüíûå âîïð...
9,9,Consumer_sentiment_2Q2025.pdf,2025,10,10 \n \nРис. 5. Ðàñïðåäåëåíèå ìíåíèé ðåñïîíäåí...


In [13]:
import pandas as pd
from data_preporation import build_page_documents

docs = build_page_documents("dataset")
tmp = pd.DataFrame(docs)

print("rows:", len(tmp))
print(tmp.groupby(["source", "year"]).size())
display(tmp[["chunk_id", "source", "year", "page"]].head(25))

rows: 21
source                                year
Consumer_sentiment_2Q2025.pdf         2025    13
Potrebitelskie_ozhidaniya_2_2019.pdf  2019     8
dtype: int64


,chunk_id,source,year,page
0,0,Consumer_sentiment_2Q2025.pdf,2025,1
1,1,Consumer_sentiment_2Q2025.pdf,2025,2
2,2,Consumer_sentiment_2Q2025.pdf,2025,3
3,3,Consumer_sentiment_2Q2025.pdf,2025,4
4,4,Consumer_sentiment_2Q2025.pdf,2025,5
5,5,Consumer_sentiment_2Q2025.pdf,2025,6
6,6,Consumer_sentiment_2Q2025.pdf,2025,7
7,7,Consumer_sentiment_2Q2025.pdf,2025,8
8,8,Consumer_sentiment_2Q2025.pdf,2025,9
9,9,Consumer_sentiment_2Q2025.pdf,2025,10


In [14]:
from agent import RAGAgent, RAGAgentAnswer, TOOLS_SCHEMA, get_tool_functions

TOOL_FUNCTIONS = get_tool_functions(table, client)

agent = RAGAgent(
    client=client,
    model=MODEL,
    tools_schema=TOOLS_SCHEMA,
    tool_functions=TOOL_FUNCTIONS,
)

print("\u2705 Агент готов")

✅ Агент готов


---
## 2. Загрузка тестового датасета

Датасет: `data/test_dataframe.xlsx` — 150 вопросов с ground truth.

| Колонка | Описание |
|---------|----------|
| `id` | Уникальный идентификатор вопроса |
| `question` | Вопрос на естественном языке |
| `ground_truth_chunks` | Эталонные чанки (строковое представление списка) |
| `document` | Год отчёта (2019 / 2025) |

In [15]:
df_test = pd.read_excel("dataset/test_dataframe.xlsx")
print(f"Размер: {df_test.shape}")
print(f"Колонки: {list(df_test.columns)}")
# df_test = df_test.head(20)
df_test.head(3)

Размер: (150, 4)
Колонки: ['id', 'question', 'relevant_text', 'document']


,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019


---
## 3. Прогон агента на тестовом датасете

Запускаем агента на каждом вопросе и сохраняем `list[RAGAgentAnswer]`.  
Это **дорогая операция** (150 вопросов × N API-вызовов), поэтому используем многопоточность.

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def _run_single(idx: int, row_id: str, question: str, agent: RAGAgent) -> tuple[int, RAGAgentAnswer]:
    """Один вызов агента (для параллельного запуска)."""
    try:
        result = agent.run(question, verbose=False, dataset_row_id=row_id)
    except Exception as e:
        print(f"  [Ошибка] вопрос {idx}: {e}")
        result = RAGAgentAnswer(dataset_row_id=row_id, answer=f"[Ошибка: {e}]", retrieved_chunks=None)
    return idx, result


def run_evaluation(df: pd.DataFrame, agent: RAGAgent, max_workers: int = 4) -> list[RAGAgentAnswer]:
    """Прогнать агента на всех вопросах (многопоточно)."""
    results: list[RAGAgentAnswer | None] = [None] * len(df)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_run_single, idx, row["id"], row["question"], agent): idx for idx, row in df.iterrows()
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Прогон агента"):
            idx, result = future.result()
            results[idx] = result

    return results

In [17]:
results = run_evaluation(df_test, agent)

print(f"\nВсего результатов: {len(results)}")
print(f"С чанками: {sum(1 for r in results if r.retrieved_chunks)}")
print(f"Без чанков (None): {sum(1 for r in results if not r.retrieved_chunks)}")

Прогон агента:   0%|          | 0/150 [00:00<?, ?it/s]


Всего результатов: 150
С чанками: 150
Без чанков (None): 0


---
## 4. Оценка через `evaluation.evaluate()`

Весь расчёт метрик — **одна строка**. Модуль `evaluation.py` принимает:
- `answers`: `list[RAGAgentAnswer]` — ответы агента
- `dataset`: `pd.DataFrame` — тестовый датасет с ground truth
- `client` + `model` — для вычисления Faithfulness (LLM-as-judge)

Возвращает `pd.DataFrame` с метриками по каждому вопросу.

In [18]:
df_test

,id,question,relevant_text,document
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,['Как мы предполагали при анализе результатов ...,2019
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...","['Одним из главных негативных факторов, тормоз...",2025
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...","['Очевидно, что основным фактором, определяющи...",2019
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,"['затрудняюсь ответить — 0,6%']",2025
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...","['В результате, баланс отрицательных и положит...",2019
...,...,...,...,...
145,17fc34cc-b8eb-49f0-b384-1d1db1998846,"Согласно данным 2025 года, какой процент респо...","['- затрудняюсь ответить — 0,3%']",2025
146,60a79f64-8f20-44dc-955e-8e8deee332e8,обобщающий индекс потребительской уверенности ...,['Обобщающий индекс потребительской уверенност...,2025
147,f5995543-80ec-4eb6-a0e2-fdf0b5d0de05,В 2025 году что является важнейшим конъюнктурн...,['Именно ожидания людей и бизнеса являются важ...,2025
148,635706a4-8be4-4550-9782-02bbaf29e2cd,Как изменилось восприятие респондентов относит...,['аспределение мнений респондентов сместилось ...,2019


In [19]:
results

[RAGAgentAnswer(dataset_row_id='e876ec3b-7442-4145-99d0-d6bb85fd982d', answer='Индекс потребительской уверенности (ИПУ) повысился на 1 процентный пункт (п. п.) до значения (-15)%.', retrieved_chunks=[Chunk(text='Потребительские настроения населения во II квартале 2019 года \n 3 \nЦентр конъюнктурных исследований Института статистических исследований и экономики знаний НИУ ВШЭ представляет информационно -аналитический материал о потребительских настроениях населения России в о II квартале 2019 г. В обзоре использованы итоги опросов по требителей, в которых принимают участие более 5 тыс. человек в возрасте от 16 лети старше, проживающих в частных домохозяйствах. Опросы проводятся Федеральной службойгосударственной статистики в ежеквартальном режиме во всех субъектах Российской Федерации. \nИндекс потребительской уверенности Росстата является важнейшей компонентойсводного индекса экономического настроения (ИЭН ВШЭ), который ежеквартально рассчитывается Центром конъюнктурных исследований и

In [32]:
from evaluation import evaluate

df_metrics = evaluate(
    answers=results,
    dataset=df_test,
    client=client,
    model=MODEL,
    overlap_threshold=0.3,
    compute_faithfulness=True,
)

print(f"\u2705 Оценка завершена: {df_metrics.shape[0]} строк, {df_metrics.shape[1]} колонок")
df_metrics.head(10)

Retrieval metrics:   0%|          | 0/150 [00:00<?, ?it/s]

Faithfulness (RAGAS):   0%|          | 0/150 [00:00<?, ?it/s]

✅ Оценка завершена: 150 строк, 9 колонок


,id,question,precision,recall,f1,mrr,map,ndcg,faithfulness
0,e876ec3b-7442-4145-99d0-d6bb85fd982d,В 2019 году как изменился индекс потребительск...,0.5,1.0,0.666667,1.0,1.0,1.0,1.000000
1,c7304cf4-9ebf-461b-acce-4dc987c21bcc,"Согласно данным 2025 года, какой фактор являет...",1.0,1.0,1.000000,1.0,1.0,1.0,1.000000
2,7e727f79-c6ff-4ff0-b101-bcec2b0f7260,"Согласно данным 2019 года, что является основн...",1.0,1.0,1.000000,1.0,1.0,1.0,1.000000
3,d52e6df3-d5ad-404e-9da9-f6f1855d16df,Какой процент респондентов в 2025 году затрудн...,1.0,1.0,1.000000,1.0,1.0,1.0,0.000000
4,536bf426-d85f-472c-b434-6796078cf647,"Согласно данным 2019 года, какой был баланс от...",1.0,1.0,1.000000,1.0,1.0,1.0,1.000000
5,e3453b4c-2d2a-4569-899d-f731158e2d36,"Согласно данным 2025 года, какие социальные по...",1.0,1.0,1.000000,1.0,1.0,1.0,1.000000
6,a3ae1b1c-a296-4e85-b8df-6ca1ed2093d0,В 2025 году какова связь между потребительским...,1.0,1.0,1.000000,1.0,1.0,1.0,0.666667
7,25a2eff0-2893-471f-a145-20aa63633e13,В 2019 году какое место в рейтинге стран ЕС по...,1.0,1.0,1.000000,1.0,1.0,1.0,1.000000
8,49287b9e-2dad-4902-a389-bc0a34494dfc,Какая доля населения в 2025 году в основном фо...,0.0,0.0,0.000000,0.0,0.0,0.0,0.500000
9,6192474e-db88-47a1-9c27-5c5e89d5e430,Какой процент респондентов оказался в затрудни...,1.0,1.0,1.000000,1.0,1.0,1.0,0.000000


---
## 5. Анализ результатов

In [33]:
metric_cols = ["precision", "recall", "f1", "mrr", "map", "ndcg", "faithfulness"]
display_names = ["Precision@K", "Recall@K", "F1@K", "MRR", "MAP", "NDCG@K", "Faithfulness"]

summary = df_metrics[metric_cols].mean()
summary.index = display_names

print("=" * 45)
print("  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА")
print("=" * 45)
for name, val in summary.items():
    bar = "\u2588" * int(val * 30) + "\u2591" * (30 - int(val * 30))
    print(f"  {name:<15} {val:.4f}  {bar}")
print("=" * 45)

  ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА RAG-АГЕНТА
  Precision@K     0.9067  ███████████████████████████░░░
  Recall@K        0.9478  ████████████████████████████░░
  F1@K            0.9144  ███████████████████████████░░░
  MRR             0.9700  █████████████████████████████░
  MAP             0.9444  ████████████████████████████░░
  NDCG@K          0.9518  ████████████████████████████░░
  Faithfulness    0.7490  ██████████████████████░░░░░░░░


In [23]:
# Сводная таблица
df_summary = pd.DataFrame(
    list(zip(display_names, summary.values)),
    columns=["Метрика", "Значение"],
)
df_summary["Значение"] = df_summary["Значение"].round(4)
df_summary

,Метрика,Значение
0,Precision@K,0.9067
1,Recall@K,0.9478
2,F1@K,0.9144
3,MRR,0.9700
4,MAP,0.9444
5,NDCG@K,0.9518
6,Faithfulness,0.0000


In [24]:
# Распределение метрик
df_metrics[metric_cols].describe().round(4)

,precision,recall,f1,mrr,map,ndcg,faithfulness
count,150.0000,150.0000,150.0000,150.0000,150.0000,150.0000,150.0
mean,0.9067,0.9478,0.9144,0.9700,0.9444,0.9518,0.0
std,0.2272,0.1893,0.2001,0.1662,0.1928,0.1794,0.0
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0
25%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
50%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
75%,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0
max,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0


In [ ]:
# Худшие вопросы по F1
worst = df_metrics.nsmallest(151, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
worst

In [ ]:
# Лучшие вопросы по F1
best = df_metrics.nlargest(100, "f1")[["id", "question", "precision", "recall", "f1", "faithfulness"]]
best

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
import os
import shutil

# Что копируем
src_folder = "/content/HSE-Agent-Systems_2026/lectures/lecture_4"   # замени на свою папку

# Куда на Google Drive
dst_folder = "/content/drive/MyDrive/ml_hw/lecture_4"

# Если папка уже есть — удалить и скопировать заново
if os.path.exists(dst_folder):
    shutil.rmtree(dst_folder)

shutil.copytree(src_folder, dst_folder)

print("Готово:", dst_folder)

Готово: /content/drive/MyDrive/ml_hw/lecture_4
